 Installing Required Libraries


In [2]:

!pip install transformers datasets pandas torch

Import Libraries

In [3]:
import transformers
import torch
import pandas as pd
print("Libraries installed successfully!")

Libraries installed successfully!


Load Email(Enron) Dataset

In [4]:
!wget https://www.cs.cmu.edu/~./enron/enron_mail_20150507.tar.gz

--2026-03-15 16:21:21--  https://www.cs.cmu.edu/~./enron/enron_mail_20150507.tar.gz
Resolving www.cs.cmu.edu (www.cs.cmu.edu)... 128.2.42.95
Connecting to www.cs.cmu.edu (www.cs.cmu.edu)|128.2.42.95|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz [following]
--2026-03-15 16:21:21--  https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz
Reusing existing connection to www.cs.cmu.edu:443.
HTTP request sent, awaiting response... 200 OK
Length: 443254787 (423M) [application/x-gzip]
Saving to: ‘enron_mail_20150507.tar.gz’

enron_mail_20150507 100%[===================>] 422.72M  1.97MB/s    in 3m 39s  

2026-03-15 16:25:00 (1.93 MB/s) - ‘enron_mail_20150507.tar.gz’ saved [443254787/443254787]



Extract Dataset

In [5]:
!tar -xzf enron_mail_20150507.tar.gz

In [6]:
import os

os.listdir("maildir")[:10]

['schoolcraft-d',
 'swerzbin-m',
 'guzman-m',
 'whitt-m',
 'sanchez-m',
 'solberg-g',
 'davis-d',
 'williams-w3',
 'mclaughlin-e',
 'smith-m']

Extract Email Body and Subject

In [7]:
import os
import pandas as pd

emails = []

for root, dirs, files in os.walk("maildir"):
    for file in files:
        path = os.path.join(root, file)
        try:
            with open(path, "r", errors="ignore") as f:
                text = f.read()

            if "Subject:" in text:
                subject = text.split("Subject:")[1].split("\n")[0]
                body = text.split("\n\n",1)[1] if "\n\n" in text else ""

                emails.append((body, subject))
        except:
            pass

df = pd.DataFrame(emails, columns=["email_body","subject"])

df.head()

,email_body,subject
0,\nPlease find attached confirmations for 02/15...,Confirmations
1,\nPlease find attached a REVISED confirmation ...,RE: Confirmations
2,\nPlease find attached confirmation of Sale of...,Border Gas Sales
3,\nPlease find attached a confirmation for 2/21...,Confirmations
4,I am doing fine! I can't wait until Friday be...,RE: Confirmation


In [8]:
df = df.sample(5000, random_state=42)

print(len(df))
df.head()


5000


,email_body,subject
427616,"Emilo,\n\nThanks for the information. Let me ...",Re: JOSE LNG Project Update
108773,"Jennifer, is there another time?\n\nRegards\nD...",Re: Meditations on Origination
355471,I think I've found what the PCA trustee may re...,PCA: Unjust Enrichment
457837,One last suggested change is in the product de...,One More change
124910,Right. This is the information that he gave t...,Re:


Prepare Input and Target Text

In [9]:
df = df.rename(columns={
    "email_body":"input_text",
    "subject":"target_text"
})

df["input_text"] = "generate subject: " + df["input_text"]

df.head()


,input_text,target_text
427616,"generate subject: Emilo,\n\nThanks for the inf...",Re: JOSE LNG Project Update
108773,"generate subject: Jennifer, is there another t...",Re: Meditations on Origination
355471,generate subject: I think I've found what the ...,PCA: Unjust Enrichment
457837,generate subject: One last suggested change is...,One More change
124910,generate subject: Right. This is the informat...,Re:


 Convert Data to HuggingFace Dataset

In [10]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

dataset

Dataset({
    features: ['input_text', 'target_text', '__index_level_0__'],
    num_rows: 5000
})

Load T5 Tokenizer and Model

In [11]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

print("Model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded


Tokenization

In [12]:

def tokenize(example):

    input_enc = tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

    target_enc = tokenizer(
        example["target_text"],
        padding="max_length",
        truncation=True,
        max_length=32
    )

    input_enc["labels"] = target_enc["input_ids"]

    return input_enc


tokenized_dataset = dataset.map(tokenize)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Model Training


In [13]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="results",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

Step,Training Loss
100,3.479191
200,1.435269
300,1.226466
400,0.979547
500,1.009899
600,0.985276
700,0.964758
800,0.964944
900,0.994496
1000,0.875471


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=1.2148819702148437, metrics={'train_runtime': 178.5029, 'train_samples_per_second': 28.011, 'train_steps_per_second': 7.003, 'total_flos': 169177251840000.0, 'train_loss': 1.2148819702148437, 'epoch': 1.0})

In [52]:
import torch
email = "Dear professor, I have completed my machine learning assignment and attached the report and code files. Please review them."
input_text = "generate subject: " + email
inputs = tokenizer.encode(input_text, return_tensors="pt")
inputs = inputs.to(model.device)
outputs = model.generate(
    inputs,
    max_length=20,
    num_beams=5,
    early_stopping=True
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Re: Update for Machine Learning


In [56]:
!pip install nltk

In [75]:
email = "Please attend the project meeting tomorrow at 10 AM"

input_text = "generate subject: " + email

inputs = tokenizer.encode(input_text, return_tensors="pt").to(model.device)

outputs = model.generate(inputs, max_length=20)

generated_subject = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Generated Subject:", generated_subject)

Generated Subject: Project Meeting


In [76]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

actual_subject = "Project Meeting Tomorrow"

reference = [actual_subject.lower().split()]
candidate = generated_subject.lower().split()

smooth = SmoothingFunction().method1

bleu = sentence_bleu(reference, candidate, smoothing_function=smooth)

print("BLEU Score:", bleu)

BLEU Score: 0.19180183554164504


In [77]:
from sklearn.metrics import precision_score, recall_score, f1_score

actual_subject = "Project Meeting Tomorrow"
generated_subject = generated_subject
true_tokens = actual_subject.lower().split()
pred_tokens = generated_subject.lower().split()

all_words = list(set(true_tokens + pred_tokens))

true_vector = [1 if word in true_tokens else 0 for word in all_words]
pred_vector = [1 if word in pred_tokens else 0 for word in all_words]
precision = precision_score(true_vector, pred_vector)
recall = recall_score(true_vector, pred_vector)
f1 = f1_score(true_vector, pred_vector)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 1.0
Recall: 0.6666666666666666
F1 Score: 0.8
